In [269]:
#imports and installs

%pip install pandas

import math
import pandas as pd
from IPython.display import display, Markdown
import time
import random

Note: you may need to restart the kernel to use updated packages.


# Questão A

## funcoes do dataframe

In [270]:
def format_scientific(value, precision):
    if value == 0: return "0.0000 x 10^0"
    
    formatted = "{:.{}e}".format(value, precision)
    
    return formatted.replace("e", " x 10^").replace("+", "")

def get_numerical_results_table(numerical_data):
    return pd.DataFrame({
        "Bisection": numerical_data["bisection"],
        "False Position": numerical_data["false_position"],
        "Fixed-Point": numerical_data["fixed_point"],
        "Newton": numerical_data["newton"],
        "Secant": numerical_data["secant"]
    }, index=[
        "Initial Data",
        "x̄",
        "f(x̄)",
        "Error in x",
        "Number of Iterations"
    ])

def get_computational_effort_table(effort_data):
    return pd.DataFrame({
        "Bisection": effort_data["bisection"],
        "False Position": effort_data["false_position"],
        "Fixed-Point": effort_data["fixed_point"],
        "Newton": effort_data["newton"],
        "Secant": effort_data["secant"]
    }, index=[
        "Operations per Iteration",
        "Operation Complexity",
        "Logical Decisions",
        "Function Evaluations per Iteration",
        "Total Number of Iterations"
    ])

def get_execution_time_table(time_data):
    return pd.DataFrame({
        "Bisection": time_data["bisection"],
        "False Position": time_data["false_position"],
        "Fixed-Point": time_data["fixed_point"],
        "Newton": time_data["newton"],
        "Secant": time_data["secant"]
    }, index=[
        "Time per Iteration (ms)",
        "Total Time (ms)"
    ])

def print_tables(numerical_data, effort_data, time_data):
    display(Markdown("## Table 1 – Numerical Results for Root-Finding Methods"))
    display(get_numerical_results_table(numerical_data))

    display(Markdown("## Table 2 – Computational Effort Analysis"))
    display(get_computational_effort_table(effort_data))

    display(Markdown("## Table 3 – Execution Time Analysis"))
    display(get_execution_time_table(time_data))

## bisseccao

In [271]:
def bisection(function, interval, stopping_crit_1, precision, max_iterations = 100):
    #(1) initial values
    a = interval[0]
    b = interval[1]
    episolon = stopping_crit_1

    #other initial values
    x = 0
    iterations = 0
    total_time = 0
    
    dyn_ops = 0
    dyn_logic = 0
    dyn_evals = 0

    #(2) first verification
    dyn_ops += 1
    dyn_logic += 1
    if (b - a) < episolon:
        x = random.uniform(a, b)

    else:
        #(3)
        iterations = 1
        
        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            dyn_evals += 1
            fa = function(a)

            #(5)
            dyn_ops += 2
            x = (a + b)/2

            dyn_evals += 1
            fx = function(x)
            
            #(6)
            dyn_ops += 1
            dyn_logic += 1
            if fa * fx > 0:
                a = x
            
            #(7)
            else:
                b = x

            #(8)
            dyn_ops += 1
            dyn_logic += 1
            if abs(b - a) < episolon:
                #not choosing a random number in the range [a, b] for accuracy
                break
            
            #(9)
            iterations += 1
        
        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        interval,                                   #initial data
        x,                                          #x̄
        format_scientific(function(x), precision),  #f(x̄)
        format_scientific(abs(b - a), precision),   #error
        iterations                                  #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,     #operations per iteration
        "O(1)",                                                   #complexity
        dyn_logic,                                                #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals, #function evals per iteration
        iterations                                                #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

## Posicao Falsa

In [272]:
def false_position(function, interval, stopping_crit_1, stopping_crit_2, precision, max_iterations = 100):
    #(1) initial values
    a = interval[0]
    b = interval[1]
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2
    
    #other initial values
    x = 0
    iterations = 0
    total_time = 0
    
    dyn_ops = 0
    dyn_logic = 0
    dyn_evals = 0

    #(2) first verification
    dyn_ops += 1
    dyn_logic += 1
    if (b - a) < episolon_1:
        x = random.uniform(a, b)
    
    else:
        dyn_evals += 1
        dyn_logic += 1
        if abs(function(a)) < episolon_2:
            x = a
        
        else:
            dyn_evals += 1
            dyn_logic += 1
            if abs(function(b)) < episolon_2:
                x = b

            else:
                #(3)
                iterations = 1

                #loop
                start_time = time.perf_counter()
                for i in range(max_iterations):
                    #(4)
                    dyn_evals += 1
                    fa = function(a)
                    dyn_evals += 1
                    fb = function(b)

                    #(5)
                    dyn_ops += 5
                    x = ((a * fb) - (b * fa))/(fb - fa)
                    
                    dyn_evals += 1
                    fx = function(x)
                    
                    #(6)
                    dyn_logic += 1
                    if abs(fx) < episolon_2:
                        break
                    
                    #(7)
                    dyn_ops += 1
                    dyn_logic += 1
                    if fa * fx > 0:
                        a = x

                    #(8)
                    else:
                        b = x

                    #(9)
                    dyn_ops += 1
                    dyn_logic += 1
                    if abs(b - a) < episolon_1:
                        x = random.uniform(a, b)
                        break

                    #(10)
                    iterations += 1
                
                #calculling the time
                final_time = time.perf_counter()
                total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        interval,                                     #initial data
        x,                                            #x̄
        format_scientific(function(x), precision),    #f(x̄)
        format_scientific(abs(b - a), precision),     #error
        iterations                                    #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,     #operations per iteration
        "O(1)",                                                   #complexity
        dyn_logic,                                                #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals, #function evals per iteration
        iterations                                                #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

## MPF

In [273]:
def fixed_point(function, iteration_function, init_x, stopping_crit_1, stopping_crit_2, precision, max_iterations = 100):
    #(1) initial values
    initial_x = init_x
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2

    #other initial values
    x = initial_x
    x_1 = 0
    iterations = 0
    total_time = 0
    current_error = 0
    
    dyn_ops = 0
    dyn_logic = 0
    dyn_evals = 0

    #(2) first verification
    dyn_evals += 1
    dyn_logic += 1
    if abs(function(initial_x)) < episolon_1:
        pass

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            dyn_evals += 1
            x_1 = iteration_function(x)
            
            dyn_evals += 1
            fx_1 = function(x_1)

            #(5)
            dyn_ops += 1
            current_error = x_1 - x
            
            dyn_logic += 2
            if abs(fx_1) < episolon_1 or abs(current_error) < episolon_2:
                x = x_1
                break
            
            #(6)
            x = x_1

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                              #initial data
        x,                                                #x̄
        format_scientific(function(x), precision),        #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,     #operations per iteration
        "O(1)",                                                   #complexity
        dyn_logic,                                                #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals, #function evals per iteration
        iterations                                                #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

## Newton

In [274]:
def get_derivative_function(function):
    def derivative(x, h=1e-8):
        return (function(x + h) - function(x - h)) / (2 * h)
    
    return derivative

def newton(function, init_x, stopping_crit_1, stopping_crit_2, precision, max_iterations = 100):
    derivative_function = get_derivative_function(function)

    #(1) initial values
    initial_x = init_x
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2

    #other initial values
    x = initial_x
    x_1 = 0
    iterations = 0
    total_time = 0
    current_error = 0
    
    dyn_ops = 0
    dyn_logic = 0
    dyn_evals = 0

    #(2) first verification
    dyn_evals += 1
    dyn_logic += 1
    if abs(function(initial_x)) < episolon_1:
        pass

    else:
        #(3)
        iterations = 1

        #loop
        start_time = time.perf_counter()
        for i in range(max_iterations):
            #(4)
            dyn_evals += 3
            dyn_ops += 2
            x_1 = x - (function(x)/derivative_function(x))
            
            dyn_evals += 1
            fx_1 = function(x_1)

            #(5)
            dyn_ops += 1
            current_error = x_1 - x
            
            dyn_logic += 2
            if abs(fx_1) < episolon_1 or abs(current_error) < episolon_2:
                x = x_1
                break
            
            #(6)
            x = x_1

            #(7)
            iterations += 1

        #calculling the time
        final_time = time.perf_counter()
        total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x}",                                #initial data
        x,                                                  #x̄
        format_scientific(function(x), precision),          #f(x̄)
        format_scientific(abs(current_error), precision),   #error
        iterations                                          #iterations
    ]

    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,      #operations per iteration
        "O(1)",                                                    #complexity
        dyn_logic,                                                 #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,  #function evals per iteration
        iterations                                                 #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

## Secant

In [275]:
def secant(function, ini_x_0, ini_x_1, stopping_crit_1, stopping_crit_2, precision, max_iterations = 100):
    #(1) initial values
    initial_x_0 = ini_x_0
    initial_x_1 = ini_x_1
    episolon_1 = stopping_crit_1
    episolon_2 = stopping_crit_2

    #other initial values
    x = initial_x_1
    x_0 = initial_x_0
    x_1 = initial_x_1
    iterations = 0
    total_time = 0
    current_error = 0
    
    dyn_ops = 0
    dyn_logic = 0
    dyn_evals = 0

    #(2) first verification
    dyn_evals += 1
    dyn_logic += 1
    if abs(function(x_0)) < episolon_1:
        x = x_0
    
    #(3) second verification
    else:
        dyn_evals += 1
        dyn_ops += 1
        dyn_logic += 2
        if abs(function(x_1)) < episolon_1 or abs(x_1 - x_0) < episolon_2:
            x = x_1
            
        else:
            #(4)
            iterations = 1

            #loop
            start_time = time.perf_counter()
            for i in range(max_iterations):
                dyn_evals += 1
                fx_0 = function(x_0)
                dyn_evals += 1
                fx_1 = function(x_1)

                #(5)
                dyn_ops += 5
                x_2 = x_1 - ((fx_1/(fx_1 - fx_0)) * (x_1 - x_0))
                
                dyn_evals += 1
                fx_2 = function(x_2)

                #(6)
                dyn_ops += 1
                current_error = x_2 - x_1
                
                dyn_logic += 2
                if abs(fx_2) < episolon_1 or abs(current_error) < episolon_2:
                    x = x_2
                    break
                
                #(7)
                x_0 = x_1
                x_1 = x_2

                #(7)
                iterations += 1

            #calculling the time
            final_time = time.perf_counter()
            total_time = (final_time - start_time) * 1000 #ms

    #datas
    #numerical data
    numerical_data = [
        f"X0 = {initial_x_0}; X1 = {initial_x_1}",        #initial data
        x,                                                #x̄
        format_scientific(function(x), precision),        #f(x̄)
        format_scientific(abs(current_error), precision), #error
        iterations                                        #iterations
    ]
    
    #effort data
    effort_data = [
        dyn_ops // iterations if iterations > 0 else dyn_ops,      #operations per iteration
        "O(1)",                                                    #complexity
        dyn_logic,                                                 #logical decisions
        dyn_evals // iterations if iterations > 0 else dyn_evals,  #function evals per iteration
        iterations                                                 #iterations
    ]

    #time data
    time_data = [
        total_time/iterations if iterations > 0 else 0, #time per iteration
        total_time                                      #total time
    ]

    return numerical_data, effort_data, time_data

## datas

In [276]:
numerical_data = {
    "bisection": [None, None, None, None, None],
    "false_position": [None, None, None, None, None],
    "fixed_point": [None, None, None, None, None],
    "newton": [None, None, None, None, None],
    "secant": [None, None, None, None, None]
}

effort_data = {
    "bisection": [None, None, None, None, None],
    "false_position": [None, None, None, None, None],
    "fixed_point": [None, None, None, None, None],
    "newton": [None, None, None, None, None],
    "secant": [None, None, None, None, None]
}

time_data = {
    "bisection": [None, None],
    "false_position": [None, None],
    "fixed_point": [None, None],
    "newton": [None, None],
    "secant": [None, None]
}

## Exemplo 18

In [277]:
example_function = lambda x: (math.e**(-x**2)) - math.cos(x)
fixed_point_iteration_function = lambda x: math.cos(x) - math.e**(-x**2) + x
interval = [1, 2]
stopping_crit_1 = stopping_crit_2 = 10**-4

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit_1, 4)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit_1, stopping_crit_2, 4)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, 1.5, stopping_crit_1, stopping_crit_2, 4)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, 1.5, stopping_crit_1, stopping_crit_2, 4)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 1, 2, stopping_crit_1, stopping_crit_2, 4)

print_tables(numerical_data, effort_data, time_data)

## Table 1 – Numerical Results for Root-Finding Methods

,Bisection,False Position,Fixed-Point,Newton,Secant
Initial Data,"[1, 2]","[1, 2]",X0 = 1.5,X0 = 1.5,X0 = 1; X1 = 2
x̄,1.447449,1.447357,1.447525,1.447416,1.447413
f(x̄),2.1921 x 10^-05,-3.6388 x 10^-05,7.0258 x 10^-05,1.3204 x 10^-06,-5.2422 x 10^-07
Error in x,6.1035 x 10^-05,5.5289 x 10^-01,1.9319 x 10^-04,1.7072 x 10^-03,1.8553 x 10^-04
Number of Iterations,14,6,6,2,5


## Table 2 – Computational Effort Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Operations per Iteration,4,6,1,3,6
Operation Complexity,O(1),O(1),O(1),O(1),O(1)
Logical Decisions,29,19,13,5,13
Function Evaluations per Iteration,2,3,2,4,3
Total Number of Iterations,14,6,6,2,5


## Table 3 – Execution Time Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Time per Iteration (ms),0.001436,0.0015,0.001,0.00215,0.00134
Total Time (ms),0.020100,0.0090,0.006,0.00430,0.00670


## Exemplo 19

In [278]:
example_function = lambda x: x**3 - x - 1
fixed_point_iteration_function = lambda x: (x + 1)**(1/3)
interval = [1, 2]
stopping_crit_1 = stopping_crit_2 = 10**-6

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit_1, 7)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit_1, stopping_crit_2, 7)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, 1, stopping_crit_1, stopping_crit_2, 7)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, 0, stopping_crit_1, stopping_crit_2, 7)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 0, .5, stopping_crit_1, stopping_crit_2, 7)

print_tables(numerical_data, effort_data, time_data)

## Table 1 – Numerical Results for Root-Finding Methods

,Bisection,False Position,Fixed-Point,Newton,Secant
Initial Data,"[1, 2]","[1, 2]",X0 = 1,X0 = 0,X0 = 0; X1 = 0.5
x̄,1.324718,1.324718,1.324718,1.324718,1.324718
f(x̄),-1.8575764 x 10^-06,-8.2906613 x 10^-07,-4.7372647 x 10^-07,2.7204905 x 10^-12,-4.3405755 x 10^-08
Error in x,9.5367432 x 10^-07,6.7528250 x 10^-01,4.7372647 x 10^-07,8.2941372 x 10^-07,1.1916614 x 10^-05
Number of Iterations,20,17,9,21,26


## Table 2 – Computational Effort Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Operations per Iteration,4,6,1,3,6
Operation Complexity,O(1),O(1),O(1),O(1),O(1)
Logical Decisions,41,52,19,43,55
Function Evaluations per Iteration,2,3,2,4,3
Total Number of Iterations,20,17,9,21,26


## Table 3 – Execution Time Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Time per Iteration (ms),0.0008,0.001059,0.000633,0.001024,0.000677
Total Time (ms),0.0160,0.018000,0.005700,0.021500,0.017600


## Exemplo 20

In [279]:
example_function = lambda x: 4 * math.sin(x) - math.e**x
fixed_point_iteration_function = lambda x: x - 2 * math.sin(x) + .5 * math.e**x
interval = [0, 1]
stopping_crit_1 = stopping_crit_2 = 10**-5

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit_1, 4)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit_1, stopping_crit_2, 4)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, .5, stopping_crit_1, stopping_crit_2, 4)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, .5, stopping_crit_1, stopping_crit_2, 4)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 0, 1, stopping_crit_1, stopping_crit_2, 4)

print_tables(numerical_data, effort_data, time_data)

## Table 1 – Numerical Results for Root-Finding Methods

,Bisection,False Position,Fixed-Point,Newton,Secant
Initial Data,"[0, 1]","[0, 1]",X0 = 0.5,X0 = 0.5,X0 = 0; X1 = 1
x̄,0.370552,0.370559,0.370556,0.370558,0.370558
f(x̄),-1.3755 x 10^-05,1.6698 x 10^-06,-4.5194 x 10^-06,-2.7836 x 10^-08,5.2605 x 10^-09
Error in x,7.6294 x 10^-06,3.7056 x 10^-01,1.6144 x 10^-05,1.3863 x 10^-04,5.7406 x 10^-06
Number of Iterations,17,8,5,3,7


## Table 2 – Computational Effort Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Operations per Iteration,4,6,1,3,6
Operation Complexity,O(1),O(1),O(1),O(1),O(1)
Logical Decisions,35,25,11,7,17
Function Evaluations per Iteration,2,3,2,4,3
Total Number of Iterations,17,8,5,3,7


## Table 3 – Execution Time Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Time per Iteration (ms),0.0011,0.001063,0.001,0.001367,0.000914
Total Time (ms),0.0187,0.008500,0.005,0.004100,0.006400


## Exemplo 21

In [280]:
example_function = lambda x: (x * math.log10(x)) - 1
fixed_point_iteration_function = lambda x: x - 1.3 * ((x * math.log10(x)) - 1)
interval = [2, 3]
stopping_crit_1 = stopping_crit_2 = 10**-7

numerical_data["bisection"], effort_data["bisection"], time_data["bisection"] = bisection(example_function, interval, stopping_crit_1, 4)
numerical_data["false_position"], effort_data["false_position"], time_data["false_position"] = false_position(example_function, interval, stopping_crit_1, stopping_crit_2, 4)
numerical_data["fixed_point"], effort_data["fixed_point"], time_data["fixed_point"] = fixed_point(example_function, fixed_point_iteration_function, 2.5, stopping_crit_1, stopping_crit_2, 4)
numerical_data["newton"], effort_data["newton"], time_data["newton"] = newton(example_function, 2.5, stopping_crit_1, stopping_crit_2, 4)
numerical_data["secant"], effort_data["secant"], time_data["secant"] = secant(example_function, 2.3, 2.7, stopping_crit_1, stopping_crit_2, 4)

print_tables(numerical_data, effort_data, time_data)

## Table 1 – Numerical Results for Root-Finding Methods

,Bisection,False Position,Fixed-Point,Newton,Secant
Initial Data,"[2, 3]","[2, 3]",X0 = 2.5,X0 = 2.5,X0 = 2.3; X1 = 2.7
x̄,2.506184,2.506184,2.506184,2.506184,2.506184
f(x̄),1.2600 x 10^-08,-9.9280 x 10^-08,2.0508 x 10^-08,1.3518 x 10^-12,2.9153 x 10^-08
Error in x,5.9605 x 10^-08,4.9382 x 10^-01,3.2006 x 10^-07,3.9882 x 10^-06,8.0561 x 10^-05
Number of Iterations,24,5,5,2,3


## Table 2 – Computational Effort Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Operations per Iteration,4,6,1,3,6
Operation Complexity,O(1),O(1),O(1),O(1),O(1)
Logical Decisions,49,16,11,5,9
Function Evaluations per Iteration,2,3,2,4,3
Total Number of Iterations,24,5,5,2,3


## Table 3 – Execution Time Analysis

,Bisection,False Position,Fixed-Point,Newton,Secant
Time per Iteration (ms),0.001033,0.00098,0.00076,0.0014,0.000833
Total Time (ms),0.024800,0.00490,0.00380,0.0028,0.002500


## Exemplo 22

In [281]:
example_22_func = lambda x: x**3 - 3.5*x**2 + 4*x - 1.5
example_22_deriv = get_derivative_function(example_22_func)
stopping_crit = 10**-7

def run_example_22_comparison():
    newton_num = {}
    newton_eff = {}
    newton_time = {}
    
    test_points = [0.5, 1.33333, 1.33334]
    test_names = ["Test 1", "Test 2", "Test 3"]

    for x0, name in zip(test_points, test_names):
        res, eff, tm = newton(example_22_func, x0, stopping_crit, stopping_crit, 4)
        
        newton_num[name] = res
        newton_eff[name] = eff
        newton_time[name] = tm

    print_example_22_tables(newton_num, newton_eff, newton_time)

def print_example_22_tables(num_data, eff_data, time_data):
    display(Markdown("## Table 1 – Numerical Results (Newton Method Comparison)"))
    df_num = pd.DataFrame(num_data, index=[
        "Initial Data", "x̄", "f(x̄)", "Error in x", "Number of Iterations"
    ])
    
    display(df_num)

    display(Markdown("## Table 2 – Computational Effort Analysis"))
    df_eff = pd.DataFrame(eff_data, index=[
        "Operations per Iteration", "Operation Complexity", 
        "Logical Decisions", "Function Evaluations per Iteration", 
        "Total Number of Iterations"
    ])

    display(df_eff)

    display(Markdown("## Table 3 – Execution Time Analysis"))
    df_time = pd.DataFrame(time_data, index=[
        "Time per Iteration (ms)", "Total Time (ms)"
    ])

    display(df_time)

run_example_22_comparison()

## Table 1 – Numerical Results (Newton Method Comparison)

,Test 1,Test 2,Test 3
Initial Data,X0 = 0.5,X0 = 1.33333,X0 = 1.33334
x̄,0.999553,0.999702,1.5
f(x̄),-9.9934 x 10^-08,-4.4473 x 10^-08,1.5345 x 10^-09
Error in x,4.4608 x 10^-04,2.9777 x 10^-04,3.9172 x 10^-05
Number of Iterations,11,35,27


## Table 2 – Computational Effort Analysis

,Test 1,Test 2,Test 3
Operations per Iteration,3,3,3
Operation Complexity,O(1),O(1),O(1)
Logical Decisions,23,71,55
Function Evaluations per Iteration,4,4,4
Total Number of Iterations,11,35,27


## Table 3 – Execution Time Analysis

,Test 1,Test 2,Test 3
Time per Iteration (ms),0.0015,0.001191,0.001041
Total Time (ms),0.0165,0.041700,0.028100


# Questão B